In [1]:
%matplotlib widget

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output


class PlotImageDigitizer:
    """
    Digitizes the evenly spaced cyan points and vertical error bars used by
    the thermal-state guessing plot.

    Calibration click order:
      1. lower-left plot corner  -> (x_min, y_min)
      2. lower-right plot corner -> (x_max, y_min)
      3. upper-left plot corner  -> (x_min, y_max)
    """

    def __init__(self):
        self.image = None
        self.clicks = []
        self.cid = None
        self.fig = None
        self.ax = None

        self.upload = widgets.FileUpload(
            accept=".png,.jpg,.jpeg,.webp",
            multiple=False,
            description="Upload plot",
        )
        self.x_min = widgets.FloatText(value=0.0, description="x min")
        self.x_max = widgets.FloatText(value=2.6, description="x max")
        self.y_min = widgets.FloatText(value=0.0, description="y min")
        self.y_max = widgets.FloatText(value=1.0, description="y max")
        self.n_points = widgets.IntText(value=50, description="points")
        self.shots = widgets.IntText(value=100, description="shots")
        self.load_button = widgets.Button(
            description="Load / recalibrate",
            button_style="primary",
        )

        self.status_out = widgets.Output()
        self.image_out = widgets.Output()
        self.results_out = widgets.Output()

        self.load_button.on_click(self._load_clicked)

        controls = widgets.VBox([
            widgets.HBox([self.upload, self.load_button]),
            widgets.HBox([self.x_min, self.x_max, self.y_min, self.y_max]),
            widgets.HBox([self.n_points, self.shots]),
        ])
        display(controls, self.status_out, self.image_out, self.results_out)

    def _uploaded_bytes(self):
        value = self.upload.value
        if not value:
            raise ValueError("Upload an image first.")

        # ipywidgets 7 returns a dict; ipywidgets 8 returns a tuple.
        item = next(iter(value.values())) if isinstance(value, dict) else value[0]
        content = item["content"] if isinstance(item, dict) else item.content
        return bytes(content)

    def _load_clicked(self, _):
        with self.status_out:
            clear_output(wait=True)
            try:
                raw = self._uploaded_bytes()
                self.image = np.asarray(
                    Image.open(io.BytesIO(raw)).convert("RGB")
                )
            except Exception as exc:
                print(f"Could not load image: {exc}")
                return

            print(
                "Click three plot corners in this order:\n"
                "1) lower-left, 2) lower-right, 3) upper-left."
            )

        self.clicks = []

        if self.cid is not None and self.fig is not None:
            self.fig.canvas.mpl_disconnect(self.cid)

        with self.image_out:
            clear_output(wait=True)
            self.fig, self.ax = plt.subplots(figsize=(14, 7))
            self.ax.imshow(self.image)
            self.ax.set_title("Click lower-left plot corner")
            self.ax.set_axis_off()
            self.cid = self.fig.canvas.mpl_connect(
                "button_press_event", self._on_click
            )
            plt.show()

        with self.results_out:
            clear_output(wait=True)

    def _on_click(self, event):
        if event.inaxes is not self.ax or event.xdata is None or event.button != 1:
            return

        self.clicks.append((float(event.xdata), float(event.ydata)))
        self.ax.plot(
            event.xdata,
            event.ydata,
            marker="o",
            linestyle="None",
            markersize=8,
        )

        prompts = [
            "Click lower-right plot corner",
            "Click upper-left plot corner",
            "Calibration complete",
        ]
        self.ax.set_title(prompts[len(self.clicks) - 1])
        self.fig.canvas.draw_idle()

        if len(self.clicks) == 3:
            self.fig.canvas.mpl_disconnect(self.cid)
            self.cid = None
            self._extract_and_show()

    @staticmethod
    def _pixel_to_y(pixel_y, y_top, y_bottom, y_min, y_max):
        return y_max - (
            (pixel_y - y_top) / (y_bottom - y_top)
        ) * (y_max - y_min)

    def _extract(self):
        if self.image is None or len(self.clicks) != 3:
            raise RuntimeError("Load an image and click all three corners first.")

        ll, lr, ul = self.clicks

        # Average coordinates that should be shared by two corner clicks.
        x_left = 0.5 * (ll[0] + ul[0])
        x_right = lr[0]
        y_bottom = 0.5 * (ll[1] + lr[1])
        y_top = ul[1]

        x_min = float(self.x_min.value)
        x_max = float(self.x_max.value)
        y_min = float(self.y_min.value)
        y_max = float(self.y_max.value)
        n_points = int(self.n_points.value)
        shots = int(self.shots.value)

        if not (x_right > x_left and y_bottom > y_top):
            raise ValueError("The three clicks do not define a valid plot rectangle.")
        if not (x_max > x_min and y_max > y_min):
            raise ValueError("Axis maximums must be larger than minimums.")
        if n_points < 2 or shots < 1:
            raise ValueError("Need at least 2 points and at least 1 shot.")

        height, width = self.image.shape[:2]
        xi0 = max(0, int(np.floor(x_left)))
        xi1 = min(width - 1, int(np.ceil(x_right)))
        yi0 = max(0, int(np.floor(y_top)))
        yi1 = min(height - 1, int(np.ceil(y_bottom)))

        roi = self.image[yi0:yi1 + 1, xi0:xi1 + 1, :3].astype(float) / 255.0
        hsv = rgb_to_hsv(roi)
        h, s, v = np.moveaxis(hsv, -1, 0)
        r, g, b = np.moveaxis(roi, -1, 0)

        # Learn the cyan marker hue from the screenshot.
        cyan_seed = (
            (s > 0.55)
            & (v > 0.65)
            & (b > r + 0.15)
            & (g > r + 0.15)
        )
        if cyan_seed.sum() < 20:
            raise RuntimeError(
                "Could not find enough cyan pixels. Check the plot-corner clicks."
            )

        marker_hue = float(np.median(h[cyan_seed]))
        hue_distance = np.minimum(
            np.abs(h - marker_hue),
            1.0 - np.abs(h - marker_hue),
        )

        # Bright/saturated pixels isolate the circular markers.
        marker_mask = (
            (hue_distance < 0.040)
            & (s > 0.55)
            & (v > 0.65)
            & ((b - r) > 0.18)
            & ((g - r) > 0.12)
        )

        # Broader mask includes dimmer error bars while rejecting axes/grid.
        errorbar_mask = (
            (hue_distance < 0.065)
            & (s > 0.20)
            & (v > 0.38)
            & ((b - r) > 0.15)
            & ((g - r) > 0.08)
        )

        pixel_x_expected = np.linspace(x_left, x_right, n_points)
        x_values = np.linspace(x_min, x_max, n_points)
        spacing = (x_right - x_left) / (n_points - 1)

        pixel_x = pixel_x_expected.copy()
        pixel_y = np.full(n_points, np.nan)
        marker_strength = np.zeros(n_points)

        x_half_window = max(3, int(round(0.35 * spacing)))
        y_half_window = max(3, int(round(0.25 * spacing)))

        # Search near each known, evenly spaced x position.
        for i, xp in enumerate(pixel_x_expected):
            local_x = xp - xi0
            xa = max(0, int(round(local_x)) - x_half_window)
            xb = min(
                marker_mask.shape[1],
                int(round(local_x)) + x_half_window + 1,
            )

            local_mask = marker_mask[:, xa:xb]
            row_counts = local_mask.sum(axis=1)
            peak_y = int(np.argmax(row_counts))
            marker_strength[i] = row_counts[peak_y]

            if row_counts[peak_y] < 2:
                continue

            ya = max(0, peak_y - y_half_window)
            yb = min(local_mask.shape[0], peak_y + y_half_window + 1)
            coordinates = np.argwhere(local_mask[ya:yb])

            yy = coordinates[:, 0] + ya
            xx = coordinates[:, 1] + xa
            weights = (s[yy, xx] * v[yy, xx]) ** 2

            pixel_y[i] = np.average(yy, weights=weights) + yi0
            pixel_x[i] = np.average(xx, weights=weights) + xi0

        # A point at exactly p=0 or p=1 can be hidden by the plot border.
        inferred_boundary = np.isnan(pixel_y)
        for i in np.flatnonzero(inferred_boundary):
            nearby = []
            for distance in range(1, n_points):
                for j in (i - distance, i + distance):
                    if 0 <= j < n_points and np.isfinite(pixel_y[j]):
                        nearby.append(pixel_y[j])
                if len(nearby) >= 2:
                    break

            reference = (
                np.mean(nearby)
                if nearby
                else 0.5 * (y_top + y_bottom)
            )
            pixel_y[i] = (
                y_bottom
                if abs(reference - y_bottom) < abs(reference - y_top)
                else y_top
            )

        error_top_pixel = np.empty(n_points)
        error_bottom_pixel = np.empty(n_points)
        error_x_half_width = max(2, int(round(0.12 * spacing)))
        allowed_gap = max(2, int(round(0.15 * spacing)))

        # Follow each vertical error bar upward and downward from the marker.
        for i, (xp, yp) in enumerate(zip(pixel_x, pixel_y)):
            if inferred_boundary[i]:
                error_top_pixel[i] = yp
                error_bottom_pixel[i] = yp
                continue

            local_x = xp - xi0
            xa = max(0, int(round(local_x)) - error_x_half_width)
            xb = min(
                errorbar_mask.shape[1],
                int(round(local_x)) + error_x_half_width + 1,
            )

            row_has_errorbar = errorbar_mask[:, xa:xb].any(axis=1)
            candidate_rows = np.flatnonzero(row_has_errorbar)

            if candidate_rows.size == 0:
                error_top_pixel[i] = yp
                error_bottom_pixel[i] = yp
                continue

            seed = candidate_rows[
                np.argmin(np.abs((candidate_rows + yi0) - yp))
            ]
            top = bottom = int(seed)

            gap = 0
            for row in range(seed - 1, -1, -1):
                if row_has_errorbar[row]:
                    top = row
                    gap = 0
                else:
                    gap += 1
                    if gap > allowed_gap:
                        break

            gap = 0
            for row in range(seed + 1, len(row_has_errorbar)):
                if row_has_errorbar[row]:
                    bottom = row
                    gap = 0
                else:
                    gap += 1
                    if gap > allowed_gap:
                        break

            error_top_pixel[i] = top + yi0
            error_bottom_pixel[i] = bottom + yi0

        y_values = np.clip(
            self._pixel_to_y(pixel_y, y_top, y_bottom, y_min, y_max),
            y_min,
            y_max,
        )
        error_high_values = np.clip(
            self._pixel_to_y(
                error_top_pixel,
                y_top,
                y_bottom,
                y_min,
                y_max,
            ),
            y_min,
            y_max,
        )
        error_low_values = np.clip(
            self._pixel_to_y(
                error_bottom_pixel,
                y_top,
                y_bottom,
                y_min,
                y_max,
            ),
            y_min,
            y_max,
        )

        result = pd.DataFrame({
            "x": x_values,
            "y": y_values,
            "yerr_low": np.maximum(0.0, y_values - error_low_values),
            "yerr_high": np.maximum(0.0, error_high_values - y_values),
            "pixel_x": pixel_x,
            "pixel_y": pixel_y,
            "marker_strength": marker_strength,
            "inferred_boundary": inferred_boundary,
        })

        # Validation column only; extracted image error bars remain separate.
        result["binomial_sigma"] = np.sqrt(
            np.clip(result["y"] * (1.0 - result["y"]) / shots, 0.0, None)
        )

        calibration = {
            "error_top_pixel": error_top_pixel,
            "error_bottom_pixel": error_bottom_pixel,
        }
        return result, calibration

    def _extract_and_show(self):
        with self.results_out:
            clear_output(wait=True)
            try:
                result, calibration = self._extract()
            except Exception as exc:
                print(f"Extraction failed: {exc}")
                return

            global digitized_df
            digitized_df = result.copy()

            # Overlay detections on the original screenshot.
            self.ax.scatter(
                result["pixel_x"],
                result["pixel_y"],
                s=70,
                facecolors="none",
            )
            self.ax.vlines(
                result["pixel_x"],
                calibration["error_top_pixel"],
                calibration["error_bottom_pixel"],
                linewidth=1.5,
            )
            self.ax.set_title("Detected points and error-bar extents")
            self.fig.canvas.draw_idle()

            inferred_count = int(result["inferred_boundary"].sum())
            print(
                f"Extracted {len(result)} points. "
                f"Inferred {inferred_count} border-hidden point(s)."
            )
            print("Results are stored in the DataFrame: digitized_df")

            fig, ax = plt.subplots(figsize=(10, 5))
            ax.errorbar(
                result["x"],
                result["y"],
                yerr=np.vstack([
                    result["yerr_low"],
                    result["yerr_high"],
                ]),
                fmt="o",
                capsize=3,
            )
            ax.set(
                xlabel="Flops",
                ylabel="Excited-state probability",
                xlim=(self.x_min.value, self.x_max.value),
                ylim=(self.y_min.value, self.y_max.value),
                title="Recreated plot from screenshot",
            )
            ax.grid(True, alpha=0.3)
            plt.show()

            display(result.round(6))


digitizer = PlotImageDigitizer()


Output()

Output()

Output()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.special import eval_laguerre
from scipy.stats import binom


eta = 0.2144
shots = int(digitizer.shots.value) if "digitizer" in globals() else 100


def thermal_weights(nbar):
    if nbar < 1e-12:
        return np.array([0]), np.array([1.0])

    q = nbar / (1 + nbar)
    nmax = max(20, int(np.ceil(np.log(1e-12) / np.log(q) - 1)))
    n = np.arange(nmax + 1)

    weights = (1 - q) * q**n
    return n, weights / weights.sum()


def model(x, nbar):
    n, weights = thermal_weights(nbar)
    rabi = np.exp(-eta**2 / 2) * eval_laguerre(n, eta**2)
    phase = np.pi * np.outer(np.asarray(x), rabi)

    return np.clip(np.cos(phase) ** 2 @ weights, 0, 1)


x = digitized_df["x"].to_numpy(float)
y = np.clip(digitized_df["y"].to_numpy(float), 0, 1)
counts = np.clip(np.rint(shots * y), 0, shots).astype(int)


def loss(z):
    nbar = np.expm1(z)
    p = np.clip(model(x, nbar), 1e-12, 1 - 1e-12)
    return -np.sum(binom.logpmf(counts, shots, p))


fit = minimize_scalar(
    loss,
    bounds=(0, np.log1p(100)),
    method="bounded"
)

nbar = np.expm1(fit.x)

print(f"nbar = {nbar:.4f}")


xx = np.linspace(x.min(), x.max(), 1000)

if {"yerr_low", "yerr_high"}.issubset(digitized_df.columns):
    yerr = np.vstack([
        digitized_df["yerr_low"].to_numpy(),
        digitized_df["yerr_high"].to_numpy()
    ])
else:
    yerr = np.sqrt(np.clip(y * (1 - y) / shots, 0, None))


plt.figure(figsize=(11, 5.5))
plt.errorbar(x, y, yerr=yerr, fmt="o", capsize=3, label="data")
plt.plot(xx, model(xx, nbar), label=rf"$\bar{{n}}={nbar:.3f}$")
plt.xlabel("Flops")
plt.ylabel("Excited-state probability")
plt.ylim(-0.03, 1.03)
plt.grid()
plt.legend()
plt.show()